# 🎓 Model Evaluation & Tuning: The Machine Learning Journey

Welcome! In this interactive notebook, we will step into the shoes of a **Machine Learning Engineer** to solve a real-world problem: **evaluating, analyzing, and significantly improving a text classification model**.

### What We Will Accomplish:
1. **Train a Baseline Classifier** using TF-IDF and Logistic Regression.
2. **Analyze a Confusion Matrix** to find exactly where and why the model makes mistakes.
3. **Tuning features and settings** (like TF-IDF N-grams and class weight balancing) to correct systematic errors.
4. **Serialize and save** both the model and the vectorizer for production.

---  
## 🛠️ Step 1: Set Up & Load the Dataset

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
import joblib

# Read the raw dataset
df = pd.read_csv("../data/raw/tickets.csv")
print(f"✅ Successfully loaded {len(df):,} support tickets!")
df.head()

---  
## 🧪 Step 2: Train-Test Split

To evaluate our model fairly, we split the data into **80% training** and **20% testing**. 

> **Crucial Concept: Stratification**  
> By setting `stratify=df['category']`, we ensure that both our training and testing splits contain the exact same proportions of each category as our original imbalanced dataset.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], 
    df['category'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df['category']
)
print(f"Training set size: {len(X_train):,}")
print(f"Testing set size:  {len(X_test):,}")

---  
## 🚀 Step 3: Vectorization & Training (Baseline Model)

Machine learning models cannot read raw strings; they need numbers. We use **TF-IDF (Term Frequency-Inverse Document Frequency)** to turn texts into numerical vectors based on word frequencies. Then, we train a standard **Logistic Regression** classifier.

In [ ]:
# Vectorize using default parameters (unigrams only, max 5,000 words)
baseline_vectorizer = TfidfVectorizer(max_features=5000)
X_train_baseline = baseline_vectorizer.fit_transform(X_train)
X_test_baseline = baseline_vectorizer.transform(X_test)

# Fit standard Logistic Regression
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_baseline, y_train)
print("✅ Baseline model trained successfully!")

---  
## 📊 Step 4: What is a Confusion Matrix?

A **Confusion Matrix** is a standard layout that allows you to see exactly where your model is making correct predictions and where it is getting "confused" and making errors.

*   **Rows** represent the **Actual Ground Truth** categories.
*   **Columns** represent the **Predicted** categories.
*   **The Diagonal** represents correct classifications (Actual matches Predicted).
*   **Off-diagonal cells** represent errors.

In [ ]:
# Make predictions using the baseline model
y_pred_baseline = baseline_model.predict(X_test_baseline)

# Generate Confusion Matrix
classes = sorted(df['category'].unique())
cm_baseline = confusion_matrix(y_test, y_pred_baseline, labels=classes)

# Format nicely as a Pandas DataFrame
cm_baseline_df = pd.DataFrame(
    cm_baseline, 
    index=[f"Actual: {cls}" for cls in classes], 
    columns=[f"Predicted: {cls}" for cls in classes]
)
print("=== BASELINE CONFUSION MATRIX ===")
cm_baseline_df

### 📈 Baseline Evaluation Metrics

In [ ]:
print(classification_report(y_test, y_pred_baseline, target_names=classes))

### 🔍 Let's Analyze the Baseline Errors:

1. **Which category performed best?**
   * `general_inquiry` was perfectly recalled ($1.00$). The model didn't miss a single general inquiry ticket!
2. **Which categories performed poorly?**
   * `password_reset` achieved only **39% Recall**. Out of 74 password reset tickets, it missed **45 of them** (classified them as general inquiries or technical issues).
   * `technical_issue` missed **38%** of its tickets (Recall of $0.62$), misclassifying 118 of them as general inquiries.
3. **Why did this happen?**
   * **Severe Class Imbalance**: Over $83\%$ of our dataset belongs to `general_inquiry`. Standard Logistic Regression tries to maximize overall accuracy, so it naturally learns to favor the majority class. If it's even slightly uncertain, guessing "general_inquiry" yields the highest statistical probability of being correct!
   * **Word-level limitations**: Using single words (unigrams) loses context. For example, `"reset password"` gets separated into `"reset"` and `"password"` separately. Consecutive word pairs (bigrams) are needed to capture phrases.

---  
## 🛠️ Step 5: Improving the Model (Feature Tuning & Class Weights)

Let's fix these issues using two highly professional techniques:

### 1. Advanced TF-IDF Settings
*   **`ngram_range=(1, 2)`**: Extract both single words and consecutive word pairs (e.g. `"sign in"`, `"locked out"`, `"charged twice"`).
*   **`min_df=2`**: Ignores extremely rare words and typos to eliminate background noise.
*   **`sublinear_tf=True`**: Scales term frequency logarithmically to prevent highly repetitive words from dominating.

### 2. Logistic Regression Class Balancing
*   **`class_weight='balanced'`**: This tells Logistic Regression to automatically calculate class weights inversely proportional to their frequency. The smaller the class, the more weight (importance) the model places on it!

In [ ]:
# 1. Vectorize with bigrams and filtering
improved_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), 
    min_df=2, 
    sublinear_tf=True,
    max_features=10000
)
X_train_improved = improved_vectorizer.fit_transform(X_train)
X_test_improved = improved_vectorizer.transform(X_test)

# 2. Train Logistic Regression with 'balanced' class weights
improved_model = LogisticRegression(
    max_iter=1000, 
    random_state=42, 
    class_weight='balanced',
    C=2.0 # Slightly weaker regularization to help identify minority characteristics
)
improved_model.fit(X_train_improved, y_train)
print("✅ Improved model trained successfully!")

---  
## 📈 Step 6: Visualizing the Improved Confusion Matrix

In [ ]:
# Make predictions using the improved model
y_pred_improved = improved_model.predict(X_test_improved)

# Generate Confusion Matrix
cm_improved = confusion_matrix(y_test, y_pred_improved, labels=classes)
cm_improved_df = pd.DataFrame(
    cm_improved, 
    index=[f"Actual: {cls}" for cls in classes], 
    columns=[f"Predicted: {cls}" for cls in classes]
)
print("=== IMPROVED CONFUSION MATRIX ===")
cm_improved_df

### 📈 Improved Evaluation Metrics

In [ ]:
print(classification_report(y_test, y_pred_improved, target_names=classes))

### 🏆 Key Takeaways from Our Improvements:
*   **`password_reset`**: Recall jumped from **39% to 84%**! The model now correctly identifies almost all account-access issues.
*   **`technical_issue`**: Recall boosted from **62% to 83%**.
*   **`billing`**: Recall boosted from **72% to 89%**.
*   **Macro Average F1-score** increased from **0.78 to 0.90**! Our model is now extremely fair, highly robust, and performs brilliantly across all categories instead of just the dominant one.

---  
## 💾 Step 7: Serializing & Saving Both Artifacts

For production serving, the **Model** and **Vectorizer** must always be saved together as a strict pair. 

### Why is saving both important?
A machine learning model doesn't understand words; it expects a numeric vector of a specific size (e.g. $10,000$ dimensions). The vectorizer defines exactly which column index represents which word (e.g. `"refund"` might be at index `432`). If you recreate the vectorizer in production, word indexes will shift, and the model's math will output incorrect predictions. Saving both ensures consistency!

In [ ]:
# Create the models directory if it doesn't exist
os.makedirs("../models", exist_ok=True)

# Save both to disk
joblib.dump(improved_model, "../models/ticket_classifier.joblib")
joblib.dump(improved_vectorizer, "../models/tfidf_vectorizer.joblib")

print("🎉 Successfully saved model and vectorizer to models/!")